# Collecting Videos from TikAPI

**Step 1 - Video Collection**

In [1]:
from tikapi import TikAPI, ValidationException, ResponseException
import json
import re
import os
import requests
import datetime
from requests.exceptions import RequestException
from retrying import retry
import numpy as np

In [2]:
#API Keys
api = TikAPI('FvjyMY0ZTgyUAKBF2WThiNoNKIdUl0QqYv1v3NtKHRB53Iwe')
User =  api.user('c_M13O1BVCA8')

In [3]:
#function to get videos by soundtrack 
import random

def fetch_posts_by_sound(api, sound_id, max_videos = 100, batch_size = 1000):
    posts = []
    cursor = None
    count = 0
    
    
    try:
        response = api.public.music(id = sound_id)
        items = response.json().get('itemList', [])
        posts.extend(items[:max_videos])
        count += len(items)
        
        #fetches videos up until the batch size
        while response and count < batch_size:
            cursor = response.json().get('cursor')
            print(f"Fetching next items for cursor: {cursor}")

            if not cursor:
                break
            

            response = response.next_items()                
            new_items = response.json().get('itemList', [])
            posts.extend(new_items)
            count += len(new_items)

        random.shuffle(posts) #shuffles videos in the batch
        return posts[:max_videos] #returns only amount specified
        
    except ValidationException as e:
        print(e, e.field)

    except ResponseException as e:
        print(e, e.response.status_code)
        
    except KeyError as e:
        print(f"Key error: {e}")
        
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    

In [4]:
#Function to download MP4 data
def download_videos_from_json(api, posts, save_directory = None):
    
    if save_directory is None:
        save_directory = os.getcwd()
    for item in posts:
        video_id = item['id']
        try:
            video_response = api.public.video(id=video_id)
            video_json = video_response.json()
            if "$other" not in video_json:
                video_response = api.public.video(id=video_id, session_id=random.randint(1,20))
                video_json = video_response.json()
            download_url = video_json['itemInfo']['itemStruct']['video']['downloadAddr'] 
            video_filename = os.path.join(save_directory, f"video_{video_id}.mp4") #saves in sepcific directory
            video_response.save_video(download_url, video_filename)
            print(f"Saved video {video_id} as {video_filename}")
        except ValidationException as e:
            print(f"Validation error for video {video_id}: {e}, {e.field}")
        except ResponseException as e:
            print(f"Response error for video {video_id}: {e}, {e.response.status_code}")
        except Exception as e:
            print(f"An unexpected error occurred for video {video_id}: {e}")

In [10]:
#fetch apple posts
apple_posts = fetch_posts_by_sound(api, '7377196584867760145', max_videos = 15, batch_size = 800)

Fetching next items for cursor: 30
Fetching next items for cursor: 60
Fetching next items for cursor: 90
Fetching next items for cursor: 120
Fetching next items for cursor: 150
Fetching next items for cursor: 180
Fetching next items for cursor: 210
Fetching next items for cursor: 240
Fetching next items for cursor: 270
Fetching next items for cursor: 300
Fetching next items for cursor: 330
Fetching next items for cursor: 360
Fetching next items for cursor: 390
Fetching next items for cursor: 420
Fetching next items for cursor: 450
Fetching next items for cursor: 480
Fetching next items for cursor: 510
Fetching next items for cursor: 540
Fetching next items for cursor: 570
Fetching next items for cursor: 600
Fetching next items for cursor: 630
Fetching next items for cursor: 660
Fetching next items for cursor: 690
Fetching next items for cursor: 720
Fetching next items for cursor: 750
Fetching next items for cursor: 780
Fetching next items for cursor: 810


In [11]:
#download apple posts
download_videos_from_json(api, apple_posts, save_directory = 'apple_videos')

Saved video 7397006982256889118 as apple_videos/video_7397006982256889118.mp4
Saved video 7394022120562412833 as apple_videos/video_7394022120562412833.mp4
Saved video 7385016422629444910 as apple_videos/video_7385016422629444910.mp4
Saved video 7393742371638332704 as apple_videos/video_7393742371638332704.mp4
Saved video 7395192604683488528 as apple_videos/video_7395192604683488528.mp4
Saved video 7398675568708947205 as apple_videos/video_7398675568708947205.mp4
Saved video 7394672744236911903 as apple_videos/video_7394672744236911903.mp4
Saved video 7392321493625015594 as apple_videos/video_7392321493625015594.mp4
Saved video 7316128472278060294 as apple_videos/video_7316128472278060294.mp4
Saved video 7397815485636594949 as apple_videos/video_7397815485636594949.mp4
Saved video 7394131712415911200 as apple_videos/video_7394131712415911200.mp4
An unexpected error occurred for video 7387147488110644522: 'downloadAddr'
Saved video 7391215434370108718 as apple_videos/video_7391215434370

In [12]:
#fetch savage posts
savage_posts = fetch_posts_by_sound(api, '6800996740322297858', max_videos = 8, batch_size = 500)

Fetching next items for cursor: 30
Fetching next items for cursor: 60
Fetching next items for cursor: 90
Fetching next items for cursor: 120
Fetching next items for cursor: 150
Fetching next items for cursor: 180
Fetching next items for cursor: 210
Fetching next items for cursor: 240
Fetching next items for cursor: 270
Fetching next items for cursor: 300
Fetching next items for cursor: 330
Fetching next items for cursor: 360
Fetching next items for cursor: 390
Fetching next items for cursor: 420
Fetching next items for cursor: 450
Fetching next items for cursor: 480
Fetching next items for cursor: 510
Fetching next items for cursor: 540
Fetching next items for cursor: 570
Fetching next items for cursor: 600
Fetching next items for cursor: 630
Fetching next items for cursor: 660


In [13]:
#download savage posts
download_videos_from_json(api, savage_posts, save_directory = 'savage_videos')

Saved video 6807431236009643270 as savage_videos/video_6807431236009643270.mp4
Saved video 6805304788972997894 as savage_videos/video_6805304788972997894.mp4
Saved video 6825667809171852549 as savage_videos/video_6825667809171852549.mp4
Saved video 6823030239480925446 as savage_videos/video_6823030239480925446.mp4
Saved video 6806009320300449030 as savage_videos/video_6806009320300449030.mp4
Saved video 6831424020085312774 as savage_videos/video_6831424020085312774.mp4
Saved video 6806779065064803589 as savage_videos/video_6806779065064803589.mp4
An unexpected error occurred for video 6808845676785470725: 'downloadAddr'


In [20]:
#fetch sayso posts
sayso_posts = fetch_posts_by_sound(api, '6763054442704145158', max_videos = 200, batch_size = 1750)

Fetching next items for cursor: 30
Fetching next items for cursor: 60
Fetching next items for cursor: 90
Fetching next items for cursor: 120
Fetching next items for cursor: 150
Fetching next items for cursor: 180
Fetching next items for cursor: 210
Fetching next items for cursor: 240
Fetching next items for cursor: 270
Fetching next items for cursor: 300
Fetching next items for cursor: 330
Fetching next items for cursor: 360
Fetching next items for cursor: 390
Fetching next items for cursor: 420
Fetching next items for cursor: 450
Fetching next items for cursor: 480
Fetching next items for cursor: 510
Fetching next items for cursor: 540
Fetching next items for cursor: 570
Fetching next items for cursor: 600
Fetching next items for cursor: 630
Fetching next items for cursor: 660
Fetching next items for cursor: 690
Fetching next items for cursor: 720
Fetching next items for cursor: 750
Fetching next items for cursor: 780
Fetching next items for cursor: 810
Fetching next items for cursor:

In [21]:
#download sayso posts
download_videos_from_json(api, sayso_posts, save_directory = 'sayso_videos')

Saved video 6778074687609097478 as sayso_videos/video_6778074687609097478.mp4
Saved video 6806022930850401542 as sayso_videos/video_6806022930850401542.mp4
Saved video 6774542061594070278 as sayso_videos/video_6774542061594070278.mp4
Saved video 6789546901952941318 as sayso_videos/video_6789546901952941318.mp4
Saved video 6840121577804270854 as sayso_videos/video_6840121577804270854.mp4
Saved video 6779322471725239557 as sayso_videos/video_6779322471725239557.mp4
Saved video 6772143219279924485 as sayso_videos/video_6772143219279924485.mp4
An unexpected error occurred for video 6774036708003417350: 'downloadAddr'
Saved video 7076501685484489990 as sayso_videos/video_7076501685484489990.mp4
Saved video 6775590570502016261 as sayso_videos/video_6775590570502016261.mp4
Saved video 6774801225314569478 as sayso_videos/video_6774801225314569478.mp4
Saved video 6801996462885555461 as sayso_videos/video_6801996462885555461.mp4
Saved video 6857485086212852997 as sayso_videos/video_6857485086212

Saved video 7190300911787347202 as sayso_videos/video_7190300911787347202.mp4
Saved video 6936222634526903557 as sayso_videos/video_6936222634526903557.mp4
Saved video 6772906111948311813 as sayso_videos/video_6772906111948311813.mp4
Saved video 6803042999933717766 as sayso_videos/video_6803042999933717766.mp4
An unexpected error occurred for video 6795202358591655174: 'downloadAddr'
Saved video 6818975902098115845 as sayso_videos/video_6818975902098115845.mp4
Saved video 6772275914999958790 as sayso_videos/video_6772275914999958790.mp4
An unexpected error occurred for video 6776140451658337542: 'downloadAddr'
Saved video 6773321966997802246 as sayso_videos/video_6773321966997802246.mp4
An unexpected error occurred for video 6788053669389651205: 'downloadAddr'
Saved video 6787921605432560902 as sayso_videos/video_6787921605432560902.mp4
Saved video 6924157940093947138 as sayso_videos/video_6924157940093947138.mp4
Saved video 7198205914896190766 as sayso_videos/video_7198205914896190766

In [ ]:
#fetch cannibal posts
cannibal_posts = fetch_posts_by_sound(api, '5000000000529843558', max_videos = 8, batch_size = 100)

In [ ]:
#download cannibal posts
download_videos_from_json(api, cannibal_posts, save_directory = 'cannibal_videos')

In [9]:
#fetch supalonely posts
supalonely_posts = fetch_posts_by_sound(api, '6759409576673560577', max_videos = 30, batch_size = 2000)

Fetching next items for cursor: 30
Fetching next items for cursor: 60
Fetching next items for cursor: 90
Fetching next items for cursor: 120
Fetching next items for cursor: 150
Fetching next items for cursor: 180
Fetching next items for cursor: 210
Fetching next items for cursor: 240
Fetching next items for cursor: 270
Fetching next items for cursor: 300
Fetching next items for cursor: 330
Fetching next items for cursor: 360
Fetching next items for cursor: 390
Fetching next items for cursor: 420
Fetching next items for cursor: 450
Fetching next items for cursor: 480
Fetching next items for cursor: 510
Fetching next items for cursor: 540
Fetching next items for cursor: 570
Fetching next items for cursor: 600
Fetching next items for cursor: 630
Fetching next items for cursor: 660
Fetching next items for cursor: 690
Fetching next items for cursor: 720
Fetching next items for cursor: 750
Fetching next items for cursor: 780
Fetching next items for cursor: 810
Fetching next items for cursor:

In [10]:
#download supalonely posts
download_videos_from_json(api, supalonely_posts, save_directory = 'supalonely_videos')

Saved video 6831918329423351046 as supalonely_videos/video_6831918329423351046.mp4
Saved video 6803821406283894021 as supalonely_videos/video_6803821406283894021.mp4
An unexpected error occurred for video 7106131356039630106: 'downloadAddr'
Saved video 6806272302674726149 as supalonely_videos/video_6806272302674726149.mp4
Saved video 6809684605143239941 as supalonely_videos/video_6809684605143239941.mp4
Saved video 6842178951872269573 as supalonely_videos/video_6842178951872269573.mp4
Saved video 6969033094540037381 as supalonely_videos/video_6969033094540037381.mp4
Saved video 6823679533586058502 as supalonely_videos/video_6823679533586058502.mp4
Saved video 6802676440761371909 as supalonely_videos/video_6802676440761371909.mp4
Saved video 6826540437394935041 as supalonely_videos/video_6826540437394935041.mp4
Saved video 6834916578438434053 as supalonely_videos/video_6834916578438434053.mp4
Saved video 6803710875296369925 as supalonely_videos/video_6803710875296369925.mp4
An unexpecte

In [10]:
#fetch nopole posts
nopole_posts = fetch_posts_by_sound(api, '7378945342212148000', max_videos = 400, batch_size = 3000)

Fetching next items for cursor: 30
Fetching next items for cursor: 60
Fetching next items for cursor: 90
Fetching next items for cursor: 120
Fetching next items for cursor: 150
Fetching next items for cursor: 180
Fetching next items for cursor: 210
Fetching next items for cursor: 240
Fetching next items for cursor: 270
Fetching next items for cursor: 300
Fetching next items for cursor: 330
Fetching next items for cursor: 360
Fetching next items for cursor: 390
Fetching next items for cursor: 420
Fetching next items for cursor: 450
Fetching next items for cursor: 480
Fetching next items for cursor: 510
Fetching next items for cursor: 540
Fetching next items for cursor: 570
Fetching next items for cursor: 600
Fetching next items for cursor: 630
Fetching next items for cursor: 660
Fetching next items for cursor: 690
Fetching next items for cursor: 720
Fetching next items for cursor: 750
Fetching next items for cursor: 780
Fetching next items for cursor: 810
Fetching next items for cursor:

In [11]:
#download nopole posts
download_videos_from_json(api, nopole_posts, save_directory = 'nopole_videos')

An unexpected error occurred for video 7432391520801721618: 'downloadAddr'
An unexpected error occurred for video 7432012343317482785: 'downloadAddr'
Saved video 7426096725116620078 as nopole_videos/video_7426096725116620078.mp4
Saved video 7439477806511181074 as nopole_videos/video_7439477806511181074.mp4
Saved video 7432421676995841323 as nopole_videos/video_7432421676995841323.mp4
An unexpected error occurred for video 7435681359819066679: 'downloadAddr'
An unexpected error occurred for video 7432722097782230289: 'downloadAddr'
An unexpected error occurred for video 7449112953619565831: 'downloadAddr'
Saved video 7439816132602662199 as nopole_videos/video_7439816132602662199.mp4
Saved video 7437865270879816978 as nopole_videos/video_7437865270879816978.mp4
An unexpected error occurred for video 7433445562105564448: 'downloadAddr'
Saved video 7442405852507589909 as nopole_videos/video_7442405852507589909.mp4
Saved video 7427153357120294175 as nopole_videos/video_7427153357120294175.m

Saved video 7429555094451997994 as nopole_videos/video_7429555094451997994.mp4
Response error for video 7429801125823417630: Something went wrong., 403
Saved video 7425094350025133358 as nopole_videos/video_7425094350025133358.mp4
Saved video 7429098565663722795 as nopole_videos/video_7429098565663722795.mp4
Saved video 7436824945818176824 as nopole_videos/video_7436824945818176824.mp4
An unexpected error occurred for video 7449548198038768938: 'downloadAddr'
Saved video 7443015938703772983 as nopole_videos/video_7443015938703772983.mp4
An unexpected error occurred for video 7475304899807300871: 'downloadAddr'
Saved video 7435624351749164320 as nopole_videos/video_7435624351749164320.mp4
An unexpected error occurred for video 7427215221900446994: 'downloadAddr'
Saved video 7447777105603054853 as nopole_videos/video_7447777105603054853.mp4
Saved video 7426815746392509738 as nopole_videos/video_7426815746392509738.mp4
An unexpected error occurred for video 7436360799695736096: 'downloadA

Saved video 7460259545986665774 as nopole_videos/video_7460259545986665774.mp4
Saved video 7439018951226477832 as nopole_videos/video_7439018951226477832.mp4
An unexpected error occurred for video 7451414645786594566: 'downloadAddr'
Saved video 7452330908067073298 as nopole_videos/video_7452330908067073298.mp4
Saved video 7442762965536754976 as nopole_videos/video_7442762965536754976.mp4
Saved video 7437623334054972728 as nopole_videos/video_7437623334054972728.mp4
Saved video 7434972119752199470 as nopole_videos/video_7434972119752199470.mp4
Saved video 7453215079769607432 as nopole_videos/video_7453215079769607432.mp4
Saved video 7443531283189288247 as nopole_videos/video_7443531283189288247.mp4
Saved video 7445223171642363144 as nopole_videos/video_7445223171642363144.mp4
Saved video 7433161010158128415 as nopole_videos/video_7433161010158128415.mp4
Saved video 7449421947152190766 as nopole_videos/video_7449421947152190766.mp4
An unexpected error occurred for video 74453709999624061

Saved video 7439418118935547167 as nopole_videos/video_7439418118935547167.mp4
Saved video 7440751482997099783 as nopole_videos/video_7440751482997099783.mp4
Saved video 7431993550885227818 as nopole_videos/video_7431993550885227818.mp4
An unexpected error occurred for video 7431006985417395462: 'downloadAddr'
Saved video 7454696521674525957 as nopole_videos/video_7454696521674525957.mp4
Saved video 7434966933507689774 as nopole_videos/video_7434966933507689774.mp4
Saved video 7431557034971073824 as nopole_videos/video_7431557034971073824.mp4
Saved video 7439245188591127839 as nopole_videos/video_7439245188591127839.mp4
Saved video 7451838824163872006 as nopole_videos/video_7451838824163872006.mp4
An unexpected error occurred for video 7446841014158920967: 'downloadAddr'
Saved video 7437489874275437856 as nopole_videos/video_7437489874275437856.mp4
Saved video 7445198392604560695 as nopole_videos/video_7445198392604560695.mp4
An unexpected error occurred for video 7439328833448332576: 